# LlamaIndex Episode 1 🦙

## Overview

* What is LlamaIndex?

        * LlamaHub (data loaders)

* How to setup Weaviate

        * Create schema


* Adding Data to Weaviate using LlamaIndex

        *  Data loader examples

* Chunking up your data

* Connecting Weaviate instance to LlamaIndex

* Simple query engine

## What is [LlamaIndex](https://www.llamaindex.ai/)?

#### Framework that enables you to connect LLMs and storage providers together seamlessly.
#### LlamaIndex 🤝 Weaviate ➡ Ultimate RAG stack

#### [LlamaHub](https://llama-hub-ui.vercel.app/): Enables you to connect to a number of external data sources (Notion, Slack, Web pages, and more!)

## Setting up Weaviate

1. Embedded 

2. WCS

3. Docker

Let's first install weaviate client and llama-index along some other dependencies:

In [2]:
%pip install -U weaviate-client llama-index llama-index-vector-stores-weaviate

/opt/homebrew/Cellar/python@3.12/3.12.5/Frameworks/Python.framework/Versions/3.12/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=42661) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


  Using cached weaviate_client-4.7.1-py3-none-any.whl.metadata (3.3 kB)
Using cached weaviate_client-4.7.1-py3-none-any.whl (368 kB)
  Attempting uninstall: weaviate-client
    Found existing installation: weaviate-client 3.26.7
    Uninstalling weaviate-client-3.26.7:
      Successfully uninstalled weaviate-client-3.26.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-readers-weaviate 0.2.0 requires weaviate-client<4.0.0,>=3.26.2, but you have weaviate-client 4.7.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Embedded

In [ ]:
import weaviate

client = weaviate.connect_to_embedded()

### WCS

In [ ]:
import weaviate
import os
  
# Set these environment variables
URL = os.getenv("WCD_URL")
APIKEY = os.getenv("WCS_API_KEY")
  
# Connect to a WCD instance
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=URL,
    auth_credentials=weaviate.auth.AuthApiKey(APIKEY)
)

### Docker

In order to run with docker, you can use our [docker configurator tool](https://weaviate.io/developers/weaviate/installation/docker-compose#configurator). 

Once you have Weaviate running with docker, you can connect using client with:

In [1]:
import weaviate
from weaviate import classes as wvc
  
# Connect to a local instance
client = weaviate.connect_to_local()

### Schema
Let's create our schema before hand, and specify a model to use

In [11]:
from weaviate import classes as wvc
# clean slate
client.collections.delete("BlogPost")

collection = client.collections.create(
    name="BlogPost",
    description="Blog post from the Weaviate website.",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_openai(
        model="text-embedding-3-small"
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-3.5-turbo"
    ),
    properties=[
        wvc.config.Property(name="text", description="Content from the blog post", data_type=wvc.config.DataType.TEXT)
    ]
)

print("Collection was created.")

INFO:httpx:HTTP Request: DELETE http://localhost:8080/v1/schema/BlogPost "HTTP/1.1 200 OK"
HTTP Request: DELETE http://localhost:8080/v1/schema/BlogPost "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8080/v1/schema "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:8080/v1/schema "HTTP/1.1 200 OK"
Collection was created.


## Adding Data to Weaviate using LlamaIndex

### SimpleDirectoryReader: Read files in your filesystem

In [12]:
from llama_index.core import SimpleDirectoryReader

blogs = SimpleDirectoryReader('./data').load_data()

### SimpleWebPageReader: Web scraper that turns HTML to text

In [ ]:
from llama_index.readers.web import SimpleWebPageReader

loader = SimpleWebPageReader()
documents = loader.load_data(urls=['https://weaviate.io/blog/llamaindex-and-weaviate'])

### NotionPageReader: Loads documents from Notion

In [ ]:
%pip install llama-index-readers-notion

In [ ]:
from llama_index.readers.notion import NotionPageReader

integration_token = ("secret_key")
page_ids = ["40be241cac924a5aa887fa85e945dbf6"]
reader = NotionPageReader(integration_token=integration_token)
documents = reader.load_data(page_ids=page_ids)

### Creating Nodes

In [13]:
from llama_index.core.node_parser import SimpleNodeParser

parser = SimpleNodeParser()
nodes = parser.get_nodes_from_documents(blogs)
print(nodes[1])

Node ID: cbfc3661-59ce-4a4b-b509-ace745ad5722
Text: Additionally, Weaviate's cross-references enable relations
between Classes, such as Users that "liked" a Product. For example,
User, Product, and Brand objects may each have a vector
representation, symbolic properties like name or price, and relations
as shown below.  ![Cross-reference](./img/Weaviate-Ref2Vec_1.png)
Ref2Vec gives Weaviate anot...


### Documents to Weaviate

In [14]:
from llama_index.vector_stores.weaviate import WeaviateVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

import openai
import os

# Lets set the OPENAI key
# os.environ["OPENAI_API_KEY"] = "sk-key"
openai.api_key = os.environ["OPENAI_API_KEY"]

# loading the documents
documents = SimpleDirectoryReader("./data/").load_data()

# If you want to load the index later, be sure to give it a name!
vector_store = WeaviateVectorStore(
    weaviate_client=client, index_name="BlogPost"
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

INFO:httpx:HTTP Request: GET http://localhost:8080/v1/schema/BlogPost "HTTP/1.1 200 OK"
HTTP Request: GET http://localhost:8080/v1/schema/BlogPost "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8080/v1/schema "HTTP/1.1 200 OK"
HTTP Request: GET http://localhost:8080/v1/schema "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://localhost:8080/v1/nodes "HTTP/1.1 200 OK"
HTTP Request: GET http://localhost:8080/v1/nodes "HTTP/1.1 200 OK"


In [15]:
collection = client.collections.get("BlogPost")
print("Objects in this collection:", len(collection))
print("Object properties example:", collection.query.fetch_objects().objects[0].properties)

INFO:httpx:HTTP Request: POST http://localhost:8080/v1/graphql "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:8080/v1/graphql "HTTP/1.1 200 OK"
Objects in this collection: 9
Object example: {'_node_type': 'TextNode', 'last_modified_date': '2024-06-05', 'text': '```javascript\nclass_obj = {\n    "class": "FoodAllergies",\n    "description": "Properties include people, their age, and allergy",\n    "invertedIndexConfig": {\n        "indexNullState": True,\n    },\n    "properties": [ ... ]\n}\n```\n\n#### Stage 3: Filtering for Specific Property State\nWe\'ve added `IsNull`, a new operator that checks if a given property is set to `null` or not. The `IsNull` operator allows you to perform queries that are filtered by the null state.\nWith `IsNull`, you can run queries that filter the results based on the null state.\n\nIn other words, you can use `IsNull` to find all objects that miss a specific property or find only those that have the property populated.\n\nReturning to our exam

### Query in LlamaIndex

In [4]:
from llama_index.vector_stores.weaviate import WeaviateVectorStore
from llama_index.core import VectorStoreIndex

vector_store = WeaviateVectorStore(
    weaviate_client=client, index_name="BlogPost"
)

loaded_index = VectorStoreIndex.from_vector_store(vector_store)

query_engine = loaded_index.as_query_engine()
response = query_engine.query("What is the intersection between LLMs and search?")
print(response)

/Users/dudanogueira/dev/weaviate/recipes/.venv/lib/python3.12/site-packages/pydantic/fields.py:814: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.8/migration/
  warn(
/Users/dudanogueira/dev/weaviate/recipes/.venv/lib/python3.12/site-packages/pydantic/fields.py:814: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.8/migration/
  warn(
/Users/dudanogueira/dev/weaviate/recipes/.venv/lib/python3.12/site-packages/pydantic/main.py:1059: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecate

The intersection between LLMs and search lies in the ability to combine vector search with linking classes through cross-references. This integration allows for the power of content-based representations for individual objects to be combined with context-based representations from relational graphs, enhancing the search experience by leveraging semantic graphs for personalization and recommendation.
